# Notebook 10: Capstone -- Transformer in JAX

## Why This Matters

Transformers are the backbone of modern AI -- GPT, Gemma, BERT, and virtually every state-of-the-art model. Implementing one from scratch in JAX demonstrates mastery of everything in this series: JAX arrays, functional purity, `jit`, `vmap`, Flax NNX modules, Optax optimizers, and lax control flow. This capstone also includes interview-ready answers to common JAX questions.


In [ ]:
%pip install -q jax jaxlib flax optax numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np
import matplotlib.pyplot as plt
import time
print("Ready.")

## 1. Multi-Head Attention as a Pure JAX Function

The attention mechanism:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Multi-head attention splits Q, K, V into H heads, computes attention independently, and concatenates:
$$\text{MHA}(X) = \text{Concat}(\text{head}_1, ..., \text{head}_H) W_O$$

Writing this as a pure JAX function (no Flax) demonstrates the functional approach.


In [ ]:
# Multi-head attention as a pure function
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / jnp.sqrt(d_k)
    if mask is not None:
        scores = jnp.where(mask, scores, -1e9)
    attn = jax.nn.softmax(scores, axis=-1)
    return attn @ V, attn

def multi_head_attention(x, W_q, W_k, W_v, W_o, n_heads):
    batch, seq_len, d_model = x.shape
    d_head = d_model // n_heads
    
    Q = (x @ W_q).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
    K = (x @ W_k).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
    V = (x @ W_v).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
    
    # Causal mask: lower triangular
    mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))
    
    # Attention: (batch, n_heads, seq_len, d_head)
    out, attn_weights = scaled_dot_product_attention(Q, K, V, mask[None, None, :, :])
    
    # Reshape and project
    out = out.swapaxes(1, 2).reshape(batch, seq_len, d_model)
    return out @ W_o, attn_weights

# Test
key = jax.random.PRNGKey(0)
batch, seq_len, d_model, n_heads = 2, 16, 64, 4
d_head = d_model // n_heads

x = jax.random.normal(key, (batch, seq_len, d_model))
W_q = jax.random.normal(key, (d_model, d_model)) * 0.02
W_k = jax.random.normal(key, (d_model, d_model)) * 0.02
W_v = jax.random.normal(key, (d_model, d_model)) * 0.02
W_o = jax.random.normal(key, (d_model, d_model)) * 0.02

out, attn = multi_head_attention(x, W_q, W_k, W_v, W_o, n_heads)
print('Input shape:      ', x.shape)
print('Output shape:     ', out.shape)
print('Attention shape:  ', attn.shape)

## 2. Transformer Encoder Block with Flax NNX

Now let us build the full encoder block using Flax NNX, combining multi-head attention + layer norm + feed-forward + residual connections.


In [ ]:
class TransformerEncoderBlock(nnx.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout_rate, rngs: nnx.Rngs):
        self.attn_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.attn_k = nnx.Linear(d_model, d_model, rngs=rngs)
        self.attn_v = nnx.Linear(d_model, d_model, rngs=rngs)
        self.attn_o = nnx.Linear(d_model, d_model, rngs=rngs)
        self.ff1 = nnx.Linear(d_model, d_ff, rngs=rngs)
        self.ff2 = nnx.Linear(d_ff, d_model, rngs=rngs)
        self.norm1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.norm2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.dropout = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.n_heads = n_heads
        self.d_model = d_model
    
    def __call__(self, x, training=True):
        batch, seq_len, d_model = x.shape
        d_head = d_model // self.n_heads
        n_heads = self.n_heads
        
        # Multi-head self-attention
        Q = self.attn_q(x).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
        K = self.attn_k(x).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
        V = self.attn_v(x).reshape(batch, seq_len, n_heads, d_head).swapaxes(1, 2)
        
        scores = Q @ K.swapaxes(-2, -1) / jnp.sqrt(d_head)
        attn = jax.nn.softmax(scores, axis=-1)
        attn_out = attn @ V
        attn_out = attn_out.swapaxes(1, 2).reshape(batch, seq_len, d_model)
        attn_out = self.attn_o(attn_out)
        
        # Residual + norm
        x = self.norm1(x + self.dropout(attn_out, deterministic=not training))
        
        # Feed-forward
        ff_out = self.ff2(jax.nn.gelu(self.ff1(x)))
        x = self.norm2(x + self.dropout(ff_out, deterministic=not training))
        
        return x

# Build and test
rngs = nnx.Rngs(params=0, dropout=1)
block = TransformerEncoderBlock(
    d_model=64, n_heads=4, d_ff=256, dropout_rate=0.1, rngs=rngs
)

x = jnp.ones((2, 16, 64))
out = block(x, training=False)
print('Encoder block output:', out.shape)

state = nnx.state(block)
n_params = sum(v.value.size for v in jax.tree_util.tree_leaves(state))
print(f'Parameters: {n_params:,}')

## 3. Training Loop on a Classification Task

A complete training loop using the transformer encoder for sequence classification.


In [ ]:
class TransformerClassifier(nnx.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_classes,
                 max_len, n_layers, dropout_rate, rngs: nnx.Rngs):
        self.embedding = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.pos_embedding = nnx.Param(
            jax.random.normal(rngs.params(), (max_len, d_model)) * 0.02
        )
        self.layers = [
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout_rate, rngs)
            for _ in range(n_layers)
        ]
        self.classifier = nnx.Linear(d_model, n_classes, rngs=rngs)
    
    def __call__(self, x, training=True):
        # x: (batch, seq_len) integer token ids
        seq_len = x.shape[1]
        h = self.embedding(x) + self.pos_embedding.value[:seq_len]
        for layer in self.layers:
            h = layer(h, training=training)
        # Mean pool over sequence
        h = h.mean(axis=1)
        return self.classifier(h)

rngs = nnx.Rngs(params=0, dropout=1)
model = TransformerClassifier(
    vocab_size=100, d_model=32, n_heads=4, d_ff=128,
    n_classes=5, max_len=32, n_layers=2, dropout_rate=0.1, rngs=rngs
)

optimizer = nnx.Optimizer(model, optax.adamw(1e-3))

@nnx.jit
def train_step(model, optimizer, x, y):
    def loss_fn(model):
        logits = model(x, training=True)
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        return -jnp.mean(log_probs[jnp.arange(len(y)), y])
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss

# Synthetic data
key = jax.random.PRNGKey(42)
N, SEQ_LEN = 2000, 16
X = jax.random.randint(key, (N, SEQ_LEN), 0, 100)
Y = jax.random.randint(key, (N,), 0, 5)

losses = []
t0 = time.time()
for epoch in range(3):
    batch_losses = []
    for i in range(0, N, 64):
        loss = train_step(model, optimizer, X[i:i+64], Y[i:i+64])
        batch_losses.append(float(loss))
    losses.extend(batch_losses)
    print(f'Epoch {epoch+1}: loss = {np.mean(batch_losses):.4f}')

print(f'Training time: {time.time()-t0:.2f}s')

## 4. JAX Interview Questions: Prepared Answers

**Q: What is jax.grad and how does it differ from loss.backward()?**

A: `jax.grad(f)` is a **function transform** -- it takes a function and returns a new function that computes the gradient. It is a pure operation with no side effects. PyTorch's `loss.backward()` mutates the `.grad` attributes of leaf tensors as a side effect. The key implication: JAX gradients compose naturally (`grad(grad(f))` is the Hessian), while PyTorch requires separate gradient tape management for higher-order derivatives.

**Q: Explain vmap.**

A: `vmap` vectorizes a function over a batch axis. Instead of writing a function that handles batches explicitly, you write it for a single example and `vmap` handles the batching. Unlike a Python for-loop, `vmap` generates fused XLA code -- it is as efficient as a manually batched implementation. The key use case: per-sample gradients via `vmap(grad(loss_fn))`.

**Q: When does jit recompile?**

A: JIT recompiles when: (1) input shapes change, (2) input dtypes change, (3) static arguments change. The compiled binary is cached keyed on (function_id, input_shapes, input_dtypes, static_args). This is why TPU training uses fixed/padded shapes -- to avoid recompilation on every unique sequence length.

**Q: What is the difference between pmap and vmap?**

A: `vmap` vectorizes a function over a batch axis **on a single device**. `pmap` runs a function **on multiple devices simultaneously**, with each device getting a different data shard. `vmap` is for batching within a device; `pmap` is for data parallelism across devices. In modern JAX, `pmap` is being replaced by `jax.sharding` + `jit`, which is more flexible for hybrid parallelism.


## 5. When Would You Choose JAX for a New Project?

Choose JAX when you have **one or more** of:

1. **TPU access** -- JAX is Google's native framework; TPU support is first-class
2. **Differential privacy** -- `vmap(grad(loss))` makes per-sample gradients trivial
3. **Custom differentiation** -- `custom_vjp`/`custom_jvp` for implicit diff, adjoint methods
4. **Research on optimization algorithms** -- composable transforms make algorithm experimentation clean
5. **Meta-learning** -- MAML, Reptile naturally expressed as `grad(grad(...))`
6. **Second-order methods** -- Hessians, NTK, Fisher information naturally composable
7. **Alignment with the JAX ecosystem** -- Gemma, Scenic, Dopamine, RLax are JAX-native

Stick with PyTorch when: production serving matters, you need dynamic shapes, team familiarity matters, or you rely on the broader PyTorch ecosystem (Detectron2, HuggingFace Transformers, etc.)


## Summary

### Key Concepts

| Topic | Implementation | Key API |
|-------|---------------|---------|
| Multi-head attention | Pure JAX function | `jnp.einsum`, `jax.nn.softmax` |
| Transformer block | Flax NNX Module | `nnx.Linear`, `nnx.LayerNorm` |
| Training loop | `nnx.jit` + `nnx.value_and_grad` | `nnx.Optimizer` |
| Interview: grad | Functional transform, pure | `jax.grad(f)` returns a function |
| Interview: vmap | Batch a single-example fn | `jax.vmap(f)` |
| Interview: jit | XLA compilation, recompiles on shape change | `@jax.jit` |

**Series complete**: You now have a solid foundation in JAX fundamentals, Flax NNX, Optax, lax control flow, multi-device training, TPU mental models, and advanced differentiation. The next step is implementing models from papers (SSMs, diffusion, etc.) in JAX.
